In [2]:
import yfinance as yf
import requests 
from io import StringIO
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import vectorbt as vbt
import talib

In c:\Users\Haresh L\anaconda3\Lib\site-packages\matplotlib\mpl-data\stylelib\bmh.mplstyle: The text.hinting_factor rcParam was deprecated in Matplotlib 3.11 and will be removed in 3.13.
In c:\Users\Haresh L\anaconda3\Lib\site-packages\matplotlib\mpl-data\stylelib\classic.mplstyle: The text.hinting_factor rcParam was deprecated in Matplotlib 3.11 and will be removed in 3.13.
In c:\Users\Haresh L\anaconda3\Lib\site-packages\matplotlib\mpl-data\stylelib\_classic_test_patch.mplstyle: The text.kerning_factor rcParam was deprecated in Matplotlib 3.11 and will be removed in 3.13.


In [3]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

response = requests.get(url, headers=headers)
response.raise_for_status()  


table = pd.read_html(StringIO(response.text))
df_tickers = table[0]


tickers = df_tickers['Symbol'].str.replace('.', '-', regex=False).tolist()


tickers = tickers[:500]

df3 = yf.download(tickers, start= "2015-05-05", end= "2026-05-05", threads = True)

try:
  close_df = df3["Close"]
  high_df = df3["High"]
  low_df = df3["Low"]
  open_df = df3["Open"]
  volume_df = df3["volume"]

except KeyError:
  
  close_df = df3.xs("Close", level=0, axis=1)
  high_df = df3.xs("High", level=0, axis=1)
  low_df = df3.xs("Low", level=0, axis=1)
  open_df = df3.xs("Open", level = 0, axis = 1)
  volume_df = df3.xs("Volume", level = 0, axis = 1)

close_df = close_df.dropna(axis=1, how="all")
high_df = high_df.dropna(axis=1, how="all")
low_df = low_df.dropna(axis=1, how="all")
open_df = open_df.dropna(axis=1, how = "all")
volume_df = volume_df.dropna(axis =1 , how = "all")

valid_cols = (
    close_df.columns.intersection(high_df.columns)
    .intersection(low_df.columns).intersection(open_df.columns).intersection(volume_df.columns)
)
close_df, high_df, low_df, open_df, volume_df = (
    close_df[valid_cols],
    high_df[valid_cols],
    low_df[valid_cols],
    open_df[valid_cols],
    volume_df[valid_cols]
)




[***********           22%                       ]  108 of 500 completed$VYLR: possibly delisted; no price data found  (1d 2015-05-05 -> 2026-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1777953600")
[**********************63%*****                  ]  316 of 500 completed$HONA: possibly delisted; no price data found  (1d 2015-05-05 -> 2026-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1777953600")
[**********************77%************           ]  384 of 500 completed$FDXF: possibly delisted; no price data found  (1d 2015-05-05 -> 2026-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1777953600")
[*********************100%***********************]  500 of 500 completed

3 Failed downloads:
['VYLR', 'HONA', 'FDXF']: possibly delisted; no price data found  (1d 2015-05-05 -> 2026-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1777953600")


### State based ADX Implementation

In [4]:
macd_dict = {}
signal_dict = {}
hist_dict = {}
rsi_dict = {}
adx_dict = {}

for ticker in close_df.columns:
    prices = close_df[ticker].to_numpy(dtype=np.float64)
    high_series = high_df[ticker]
    low_series = low_df[ticker]
    volume_values = volume_df[ticker].values

    macd, signal, hist = talib.MACD(
        prices,
        fastperiod=12,
        slowperiod=26,
        signalperiod=9
    )

    rsi = talib.RSI(
        prices,
        timeperiod=14
    )

    adx = talib.ADX(high_series.values, low_series.values, prices, timeperiod=14)


    macd_dict[ticker] = macd
    signal_dict[ticker] = signal
    hist_dict[ticker] = hist
    rsi_dict[ticker] = rsi
    adx_dict[ticker] = adx

macd_df = pd.DataFrame(macd_dict, index=close_df.index)
signal_df = pd.DataFrame(signal_dict, index=close_df.index)
hist_df = pd.DataFrame(hist_dict, index=close_df.index)
rsi_df = pd.DataFrame(rsi_dict, index = close_df.index )
adx_df = pd.DataFrame(adx_dict, index = close_df.index)


entries = (adx_df >= 25).shift(1).fillna(False).astype(bool)
exits = (adx_df < 20).shift(1).fillna(False).astype(bool)


backtest1 = vbt.Portfolio.from_signals(
    close=close_df,
    open = open_df,
    high = high_df,
    low = low_df,
    entries=entries,
    exits=exits,
    init_cash=100_000.0,
    size=0.1,                         
    size_type='percent',
    cash_sharing=True,                 
    group_by=True,                     
    freq='1D',
    fees=0.001,
    sl_stop = 0.10,
       
)

print(backtest1.stats())
backtest1.plot().show()

equity = backtest1.value(group_by=True)
equity.vbt.plot().show()

Start                             2015-05-05 00:00:00
End                               2026-05-04 00:00:00
Period                             2766 days 00:00:00
Start Value                                  100000.0
End Value                               469976.331706
Total Return [%]                           369.976332
Benchmark Return [%]                       524.559281
Max Gross Exposure [%]                          100.0
Total Fees Paid                          33052.772612
Max Drawdown [%]                            38.482965
Max Drawdown Duration               273 days 00:00:00
Total Trades                                    21554
Total Closed Trades                             21348
Total Open Trades                                 206
Open Trade PnL                           30097.602911
Win Rate [%]                                50.374742
Best Trade [%]                             926.788924
Worst Trade [%]                             -47.70371
Avg Winning Trade [%]       

c:\Users\Haresh L\anaconda3\Lib\site-packages\vectorbt\generic\plots_builder.py:329: UserWarning:

Subplot 'orders' does not support grouped data

c:\Users\Haresh L\anaconda3\Lib\site-packages\vectorbt\generic\plots_builder.py:329: UserWarning:

Subplot 'trade_pnl' does not support grouped data



### Crossover ADX strategy

In [4]:
entries = (
    (adx_df >= 25) &
    (adx_df.shift(1) < 25)
).shift(1).fillna(False).astype(bool)

exits = (
    (adx_df < 20) &
    (adx_df.shift(1) >= 20)
).shift(1).fillna(False).astype(bool)


backtest1 = vbt.Portfolio.from_signals(
    close=close_df,
    open = open_df,
    high = high_df,
    low = low_df,
    entries=entries,
    exits=exits,
    init_cash=100_000.0,
    size=0.1,                         
    size_type='percent',
    cash_sharing=True,                 
    group_by=True,                     
    freq='1D',
    fees=0.001,
    sl_stop = 0.10,
       
)

print(backtest1.stats())
backtest1.plot().show()

Start                             2015-05-05 00:00:00
End                               2026-05-04 00:00:00
Period                             2766 days 00:00:00
Start Value                                  100000.0
End Value                               329693.840051
Total Return [%]                            229.69384
Benchmark Return [%]                       524.559252
Max Gross Exposure [%]                      99.999636
Total Fees Paid                          25008.285807
Max Drawdown [%]                            38.145888
Max Drawdown Duration               382 days 00:00:00
Total Trades                                    16684
Total Closed Trades                             16524
Total Open Trades                                 160
Open Trade PnL                           20780.085151
Win Rate [%]                                51.537158
Best Trade [%]                             926.788924
Worst Trade [%]                            -35.872511
Avg Winning Trade [%]       

c:\Users\Haresh L\anaconda3\Lib\site-packages\vectorbt\generic\plots_builder.py:329: UserWarning:

Subplot 'orders' does not support grouped data

c:\Users\Haresh L\anaconda3\Lib\site-packages\vectorbt\generic\plots_builder.py:329: UserWarning:

Subplot 'trade_pnl' does not support grouped data



### State based ADX strategy with RSI and MACD combination

In [5]:
entries = ((adx_df >= 25) & (macd_df >= signal_df) & (rsi_df >= 30)).shift(1).fillna(False).astype(bool)
exits = ((adx_df < 20)| (macd_df < signal_df) | (rsi_df > 70)).shift(1).fillna(False).astype(bool)


backtest1 = vbt.Portfolio.from_signals(
    close=close_df,
    open = open_df,
    high = high_df,
    low = low_df,
    entries=entries,
    exits=exits,
    init_cash=100_000.0,
    size=0.1,                         
    size_type='percent',
    cash_sharing=True,                 
    group_by=True,                     
    freq='1D',
    fees=0.001,
    sl_stop = 0.10,
       
)

print(backtest1.stats())
backtest1.plot().show()

Start                             2015-05-05 00:00:00
End                               2026-05-04 00:00:00
Period                             2766 days 00:00:00
Start Value                                  100000.0
End Value                               215963.828825
Total Return [%]                           115.963829
Benchmark Return [%]                       524.559252
Max Gross Exposure [%]                      99.999861
Total Fees Paid                           67282.30063
Max Drawdown [%]                            25.557424
Max Drawdown Duration               483 days 00:00:00
Total Trades                                    27586
Total Closed Trades                             27493
Total Open Trades                                  93
Open Trade PnL                            8886.651108
Win Rate [%]                                46.211763
Best Trade [%]                             230.900287
Worst Trade [%]                            -44.096845
Avg Winning Trade [%]       

c:\Users\Haresh L\anaconda3\Lib\site-packages\vectorbt\generic\plots_builder.py:329: UserWarning:

Subplot 'orders' does not support grouped data

c:\Users\Haresh L\anaconda3\Lib\site-packages\vectorbt\generic\plots_builder.py:329: UserWarning:

Subplot 'trade_pnl' does not support grouped data

